# Classification: hyperparameter tuning and cross-validation

Companion to `model_classification_dev.ipynb`. It uses the same features (`NUM_COLS + CAT_COLS`), the same as-of-run-date labels, the same run date (2018-06-02) and the same five classifiers. It adds:

1. **A chronological hold-out.** The newest 30 days of the training window are the test set. This replaces the random stratified split.
2. **Time-series cross-validation** on the training part. Each fold trains on earlier approval days and validates on later ones.
3. **Hyperparameter tuning** with `RandomizedSearchCV` on those folds, optimising **PR-AUC** (average precision). PR-AUC is the brief's recommended metric for an imbalanced target (~13% late).
4. **Threshold selection.** The late/on-time cut-off is chosen from out-of-fold CV scores (max F1) instead of a fixed 0.5.
5. **Default vs tuned, compared on:**
   - the hold-out;
   - a pooled monthly backtest (April–July 2018) that follows the operational flow of `model_classification_dev.ipynb`.

The tuned parameters and thresholds are saved to `results/classification_tuning.json`, so `model_classification_dev.ipynb` can load them.

# Updates
- 20261001 - Initial tuning/CV notebook

# Environment set-up

In [1]:
import json, os, sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('.') if os.path.isdir('src') else os.path.abspath('phase2'))

import numpy as np
import pandas as pd
from IPython.display import display
from joblib import Parallel, delayed
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold

from src import RANDOM_STATE, LOOKBACK, PXD
from src.data import load_olist_data
from src.features import build_feature_table, NUM_COLS, CAT_COLS, LEAKAGE_COLS
from src.labels import get_required_dates, labels_as_of, attach_prediction_labels
from src.splits import chronological_split, day_blocked_time_series_folds
from src.evaluation import classification_metrics
from src.tuning import (build_classifiers, cv_summary, tune_classifier, out_of_fold_scores,
                        best_f1_threshold, params_to_json, CLASSIFIER_SEARCH_SPACES, MODEL_FAMILY)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.3f}'.format)

# Constants

In [2]:
RUN_DATE = '2018-06-02'   # Same run date as model_classification_dev.ipynb
TEST_DAYS = 30            # Newest approval days held out (~1 month, as in the dev notebook)
N_CV_FOLDS = 5
# Random-search iterations per model; kept small for laptop run times (~10 min on 4 cores)
N_ITER = {'Logistic Regression': 12, 'Decision Tree': 15, 'Random Forest': 8,
          'XGBoost': 12, 'LightGBM': 12}
FEATURE_COLS = NUM_COLS + CAT_COLS
LABEL_COL = 'dpt_var_late_delivery'
assert not set(FEATURE_COLS) & set(LEAKAGE_COLS), 'Outcome columns used as features'

# Load data and training window

In [3]:
olist = load_olist_data(verbose=False)
orders = olist['orders']
final_df = build_feature_table(olist)

start_date, end_date, inference_date = get_required_dates(RUN_DATE, lookback=LOOKBACK, pXd=PXD)
window = labels_as_of(final_df[final_df['order_approved_dt'].between(start_date, end_date)], RUN_DATE)
labelled = window[window['label_as_of_run'].notna()].copy()
labelled[LABEL_COL] = labelled['label_as_of_run'].astype(int)
print(f'Known labels: {len(labelled):,}/{len(window):,}; late rate {labelled[LABEL_COL].mean():.2%}')
assert len(labelled) == 39_956, 'Should match model_classification_dev.ipynb'

Train-test period for the inference date of 2018-06-01 with buffer of 45 days:
    2017-10-20 to 2018-04-17 (180 days)
Known labels: 39,956/39,960; late rate 13.42%


# Chronological split and CV folds
- **Test:** the last 30 approval days of the window.
- **Train:** everything before that.
- **CV folds:** cut by approval day on the training part only, with an expanding window. Fold *k* trains on blocks 1…*k* and validates on block *k*+1. No approval day is split across train and validation.

**Why not a random stratified split?** The dev notebook uses one, so its test set is spread across the whole window and the model trains on orders *after* some of the orders it is tested on. In operation the model only ever scores *future* orders. Time-ordered splits reproduce that, so their scores are what we can honestly expect.

Each part's late rate is shown below, since we don't stratify. The comparison further down measures how much the random split flatters the scores.

In [4]:
train_df, test_df = chronological_split(labelled, date_col='order_approved_dt', test_days=TEST_DAYS)
train_df = train_df.sort_values('order_approved_dt', kind='stable')
X_train, y_train = train_df[FEATURE_COLS], train_df[LABEL_COL]
X_test, y_test = test_df[FEATURE_COLS], test_df[LABEL_COL]
print(f'Train: {train_df["order_approved_dt"].min():%Y-%m-%d} to {train_df["order_approved_dt"].max():%Y-%m-%d} '
      f'({len(train_df):,} orders, {y_train.mean():.2%} late)')
print(f'Test:  {test_df["order_approved_dt"].min():%Y-%m-%d} to {test_df["order_approved_dt"].max():%Y-%m-%d} '
      f'({len(test_df):,} orders, {y_test.mean():.2%} late)')

cv_folds = day_blocked_time_series_folds(train_df['order_approved_dt'], n_splits=N_CV_FOLDS)
display(pd.DataFrame([{
    'fold': k, 'train_orders': len(tr), 'valid_orders': len(va),
    'valid_from': train_df['order_approved_dt'].iloc[va].min().date(),
    'valid_to': train_df['order_approved_dt'].iloc[va].max().date(),
    'valid_late_rate': y_train.iloc[va].mean(),
} for k, (tr, va) in enumerate(cv_folds, 1)]).set_index('fold'))

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
default_models = build_classifiers(NUM_COLS, CAT_COLS, scale_pos_weight)

Train: 2017-10-20 to 2018-03-18 (33,086 orders, 14.05% late)
Test:  2018-03-19 to 2018-04-17 (6,870 orders, 10.38% late)


,train_orders,valid_orders,valid_from,valid_to,valid_late_rate
fold,,,,,
1,3663,7599,2017-11-14,2017-12-08,0.168
2,11262,3863,2017-12-09,2018-01-02,0.078
3,15125,6142,2018-01-03,2018-01-27,0.081
4,21267,5717,2018-01-28,2018-02-21,0.123
5,26984,6102,2018-02-22,2018-03-18,0.260


# Cross-validated defaults
These are the untuned settings from `model_classification_dev.ipynb`, scored on the time-series folds:
- **PR-AUC**: the primary metric. A random model scores the late rate, about 0.13.
- **ROC-AUC**.
- **F1** at the default 0.5 threshold.

In [5]:
cv_rows = []
for name, model in default_models.items():
    cv_rows.append({'model': name, 'variant': 'default', 'family': MODEL_FAMILY[name],
                    **cv_summary(model, X_train, y_train, cv_folds)})
display(pd.DataFrame(cv_rows).set_index(['model', 'variant']))

,,family,cv_pr_auc_mean,cv_pr_auc_std,cv_roc_auc_mean,cv_roc_auc_std,cv_f1_at_0.5_mean,cv_f1_at_0.5_std
model,variant,,,,,,,
Logistic Regression,default,Linear,0.244,0.091,0.651,0.027,0.286,0.091
Decision Tree,default,Tree-based,0.229,0.094,0.603,0.041,0.279,0.094
Random Forest,default,Tree-based,0.298,0.079,0.655,0.029,0.267,0.098
XGBoost,default,Tree-based,0.293,0.079,0.641,0.028,0.253,0.091
LightGBM,default,Tree-based,0.291,0.075,0.642,0.034,0.256,0.101


## How much does a random split flatter the scores?
This scores default LightGBM with stratified *random* 5-fold CV on the same training rows. The difference from the time-series CV above is the optimism we would report by shuffling future orders into training.

In [6]:
random_cv = cv_summary(default_models['LightGBM'], X_train, y_train,
                       StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE))
time_cv = next(r for r in cv_rows if r['model'] == 'LightGBM')
display(pd.DataFrame({'random stratified 5-fold': random_cv, 'time-series 5-fold': time_cv})
        .loc[['cv_pr_auc_mean', 'cv_roc_auc_mean', 'cv_f1_at_0.5_mean']])

,random stratified 5-fold,time-series 5-fold
cv_pr_auc_mean,0.427,0.291
cv_roc_auc_mean,0.761,0.642
cv_f1_at_0.5_mean,0.411,0.256


# Hyperparameter tuning
Each model runs a random search over a small grid (`src/tuning.py`, `CLASSIFIER_SEARCH_SPACES`):
- It uses the same time-series folds and maximises mean PR-AUC.
- The class-imbalance weighting stays as in the dev notebook: `class_weight='balanced'`, or `scale_pos_weight` for the boosting models.
- The test set is never used for selection.

In [7]:
tuned_models, best_params = {}, {}
for name, model in default_models.items():
    tuned, params, search = tune_classifier(model, CLASSIFIER_SEARCH_SPACES[name], X_train, y_train,
                                            cv_folds, n_iter=N_ITER[name])
    tuned_models[name], best_params[name] = tuned, params
    cv_rows.append({'model': name, 'variant': 'tuned', 'family': MODEL_FAMILY[name],
                    **cv_summary(tuned, X_train, y_train, cv_folds)})
    print(f'{name}: {params_to_json(params)}')

cv_results = pd.DataFrame(cv_rows).set_index(['model', 'variant']).sort_index()
display(cv_results)
gain = (cv_results.xs('tuned', level='variant')['cv_pr_auc_mean']
        - cv_results.xs('default', level='variant')['cv_pr_auc_mean'])
display(gain.rename('cv_pr_auc_gain_from_tuning').to_frame())

Logistic Regression: {'classifier__C': 100.0}


Decision Tree: {'classifier__min_samples_leaf': 50, 'classifier__max_depth': 8, 'classifier__criterion': 'gini'}


Random Forest: {'classifier__n_estimators': 100, 'classifier__min_samples_leaf': 5, 'classifier__max_features': 'sqrt', 'classifier__max_depth': 10}


XGBoost: {'classifier__subsample': 0.7, 'classifier__n_estimators': 100, 'classifier__min_child_weight': 1, 'classifier__max_depth': 3, 'classifier__learning_rate': 0.05, 'classifier__colsample_bytree': 0.6}


LightGBM: {'classifier__subsample_freq': 1, 'classifier__subsample': 0.7, 'classifier__reg_lambda': 0.0, 'classifier__num_leaves': 15, 'classifier__n_estimators': 200, 'classifier__min_child_samples': 50, 'classifier__learning_rate': 0.03, 'classifier__colsample_bytree': 1.0}


family  cv_pr_auc_mean  cv_pr_auc_std  \
model               variant                                              
Decision Tree       default  Tree-based           0.229          0.094   
                    tuned    Tree-based           0.258          0.078   
LightGBM            default  Tree-based           0.291          0.075   
                    tuned    Tree-based           0.306          0.073   
Logistic Regression default      Linear           0.244          0.091   
                    tuned        Linear           0.258          0.065   
Random Forest       default  Tree-based           0.298          0.079   
                    tuned    Tree-based           0.299          0.077   
XGBoost             default  Tree-based           0.293          0.079   
                    tuned    Tree-based           0.305          0.079   

                             cv_roc_auc_mean  cv_roc_auc_std  \
model               variant                                    
Decision Tree       default            0.603           0.041   
                    tuned              0.623           0.037   
LightGBM            default            0.642           0.034   
                    tuned              0.660           0.021   
Logistic Regression default            0.651           0.027   
                    tuned              0.654           0.036   
Random Forest       default            0.655           0.029   
                    tuned              0.660           0.026   
XGBoost             default            0.641           0.028   
                    tuned              0.666           0.014   

                             cv_f1_at_0.5_mean  cv_f1_at_0.5_std  
model               variant                                       
Decision Tree       default              0.279             0.094  
                    tuned                0.267             0.086  
LightGBM            default              0.256             0.101  
                    tuned                0.261             0.097  
Logistic Regression default              0.286             0.091  
                    tuned                0.278             0.061  
Random Forest       default              0.267             0.098  
                    tuned                0.278             0.094  
XGBoost             default              0.253             0.091  
                    tuned                0.243             0.096

,cv_pr_auc_gain_from_tuning
model,
Decision Tree,0.029
LightGBM,0.015
Logistic Regression,0.013
Random Forest,0.001
XGBoost,0.012


# Threshold selection
At 0.5 the balanced models flag far more orders than are actually late, so precision is low.
- For each tuned model, this picks the threshold that maximises F1 on the **out-of-fold** CV scores.
- Each score comes from a model that never saw that order, so choosing the threshold doesn't leak information.
- Operations may instead prefer a capacity rule (e.g. contact the top 10% riskiest orders each day). The decile coverage tables in the dev notebook support that.

In [8]:
thresholds = {}
for name, model in tuned_models.items():
    oof = out_of_fold_scores(model, X_train, y_train, cv_folds)
    thresholds[name], oof_f1 = best_f1_threshold(y_train, oof)
    print(f'{name}: threshold {thresholds[name]:.3f} (out-of-fold F1 {oof_f1:.3f})')

Logistic Regression: threshold 0.453 (out-of-fold F1 0.288)


Decision Tree: threshold 0.467 (out-of-fold F1 0.327)


Random Forest: threshold 0.457 (out-of-fold F1 0.340)


XGBoost: threshold 0.357 (out-of-fold F1 0.326)


LightGBM: threshold 0.383 (out-of-fold F1 0.330)


# Hold-out evaluation (newest 30 days of the window)

In [9]:
holdout_rows = []
for variant, model_set in [('default', default_models), ('tuned', tuned_models)]:
    for name, model in model_set.items():
        fitted = clone(model).fit(X_train, y_train)
        prob = fitted.predict_proba(X_test)[:, list(fitted.classes_).index(1)]
        cutoffs = [('0.5', 0.5)] + ([('CV-tuned', thresholds[name])] if variant == 'tuned' else [])
        for label, cutoff in cutoffs:
            metrics = classification_metrics(y_test, (prob >= cutoff).astype(int), prob)
            holdout_rows.append({'model': name, 'variant': variant, 'threshold': label,
                                 'flagged_pct': (prob >= cutoff).mean(), **metrics})
holdout_results = pd.DataFrame(holdout_rows).set_index(['model', 'variant', 'threshold'])
display(holdout_results[['pr_auc', 'roc_auc', 'precision', 'recall', 'f1_score',
                         'balanced_accuracy', 'flagged_pct']])

pr_auc  roc_auc  precision  recall  \
model               variant threshold                                       
Logistic Regression default 0.5         0.214    0.695      0.177   0.633   
Decision Tree       default 0.5         0.184    0.666      0.126   0.840   
Random Forest       default 0.5         0.208    0.702      0.150   0.802   
XGBoost             default 0.5         0.216    0.685      0.132   0.853   
LightGBM            default 0.5         0.218    0.692      0.133   0.860   
Logistic Regression tuned   0.5         0.219    0.703      0.159   0.752   
                            CV-tuned    0.219    0.703      0.135   0.854   
Decision Tree       tuned   0.5         0.191    0.689      0.130   0.881   
                            CV-tuned    0.191    0.689      0.130   0.881   
Random Forest       tuned   0.5         0.206    0.704      0.151   0.802   
                            CV-tuned    0.206    0.704      0.139   0.844   
XGBoost             tuned   0.5         0.205    0.695      0.128   0.882   
                            CV-tuned    0.205    0.695      0.110   0.973   
LightGBM            tuned   0.5         0.209    0.699      0.130   0.879   
                            CV-tuned    0.209    0.699      0.116   0.941   

                                       f1_score  balanced_accuracy  \
model               variant threshold                                
Logistic Regression default 0.5           0.277              0.646   
Decision Tree       default 0.5           0.219              0.582   
Random Forest       default 0.5           0.252              0.637   
XGBoost             default 0.5           0.229              0.602   
LightGBM            default 0.5           0.231              0.606   
Logistic Regression tuned   0.5           0.263              0.646   
                            CV-tuned      0.233              0.609   
Decision Tree       tuned   0.5           0.227              0.599   
                            CV-tuned      0.227              0.599   
Random Forest       tuned   0.5           0.254              0.639   
                            CV-tuned      0.239              0.620   
XGBoost             tuned   0.5           0.224              0.593   
                            CV-tuned      0.198              0.531   
LightGBM            tuned   0.5           0.226              0.598   
                            CV-tuned      0.207              0.556   

                                       flagged_pct  
model               variant threshold               
Logistic Regression default 0.5              0.371  
Decision Tree       default 0.5              0.694  
Random Forest       default 0.5              0.556  
XGBoost             default 0.5              0.670  
LightGBM            default 0.5              0.670  
Logistic Regression tuned   0.5              0.489  
                            CV-tuned         0.659  
Decision Tree       tuned   0.5              0.703  
                            CV-tuned         0.703  
Random Forest       tuned   0.5              0.553  
                            CV-tuned         0.629  
XGBoost             tuned   0.5              0.715  
                            CV-tuned         0.918  
LightGBM            tuned   0.5              0.704  
                            CV-tuned         0.840

# Pooled monthly backtest (operational flow)
This mirrors the backtest in `model_classification_dev.ipynb`:
1. For each month, fit on the 2nd using the full labelled window as of that date.
2. Score every order approved that month that wasn't delivered by the end of its approval day.
3. Attach each order's actual label at approval day + 45 days.

Default and tuned models are both compared, with the tuned ones at their CV threshold.

**Months: April–July 2018.** The hyperparameters and thresholds were chosen using approvals up to 2018-03-18, so these months were never used for tuning. Earlier months would reuse the tuning data and look better than they should.

In [10]:
BACKTEST_RUN_DATES = ['2018-04-02', '2018-05-02', '2018-06-02', '2018-07-02']

def backtest_month(run_date):
    start, end, _ = get_required_dates(run_date, lookback=LOOKBACK, pXd=PXD, verbose=False)
    history = labels_as_of(final_df[final_df['order_approved_dt'].between(start, end)], run_date)
    history = history[history['label_as_of_run'].notna()]
    y_fit = history['label_as_of_run'].astype(int)
    month = pd.Period(run_date, 'M')
    cohort = final_df[final_df['order_approved_dt'].dt.to_period('M').eq(month)]
    delivered_day = cohort['order_delivered_customer_date'].dt.normalize()
    cohort = cohort[delivered_day.isna() | delivered_day.gt(cohort['order_approved_dt'])]

    ratio = (y_fit == 0).sum() / (y_fit == 1).sum()
    defaults = build_classifiers(NUM_COLS, CAT_COLS, ratio)
    frames = []
    for variant in ['default', 'tuned']:
        for name, model in defaults.items():
            model = clone(model).set_params(**best_params[name]) if variant == 'tuned' else clone(model)
            model.fit(history[FEATURE_COLS], y_fit)
            frame = cohort[['order_id', 'order_approved_dt']].copy()
            frame['predicted_probability'] = model.predict_proba(cohort[FEATURE_COLS])[:, 1]
            frame['model'], frame['variant'], frame['month'] = name, variant, str(month)
            frames.append(frame)
    return pd.concat(frames, ignore_index=True)

backtest = pd.concat(Parallel(n_jobs=-1)(delayed(backtest_month)(d) for d in BACKTEST_RUN_DATES),
                     ignore_index=True)
labels = attach_prediction_labels(backtest[['order_id']].drop_duplicates()
                                  .assign(predicted_probability=0.0), orders, lookback=LOOKBACK)
backtest = backtest.merge(labels[['order_id', 'actual_label', 'label_status']], on='order_id')
print(f'{backtest["order_id"].nunique():,} scored orders; '
      f'known labels {backtest.drop_duplicates("order_id")["actual_label"].notna().mean():.2%}')

26,092 scored orders; known labels 99.94%


In [11]:
def backtest_metrics(group):
    known = group[group['actual_label'].notna()]
    y = known['actual_label'].astype(int)
    name, variant = group.name[-2], group.name[-1]
    cutoff = thresholds[name] if variant == 'tuned' else 0.5
    return pd.Series(classification_metrics(y, (known['predicted_probability'] >= cutoff).astype(int),
                                            known['predicted_probability']))

monthly_metrics = (backtest.groupby(['month', 'model', 'variant'])
                   .apply(backtest_metrics).reset_index())
late_rate = (backtest.drop_duplicates('order_id').dropna(subset=['actual_label'])
             .groupby('month')['actual_label'].mean().astype(float))
print('Actual late rate per month (a random model scores PR-AUC equal to this):')
display(late_rate.to_frame('late_rate').T)
print('PR-AUC by month:')
display(monthly_metrics.pivot_table(index=['model', 'variant'], columns='month', values='pr_auc'))

summary = monthly_metrics.groupby(['model', 'variant'])[
    ['pr_auc', 'roc_auc', 'precision', 'recall', 'f1_score']].agg(['mean', 'std'])
pooled = (backtest.groupby(['model', 'variant']).apply(backtest_metrics)
          [['pr_auc', 'precision', 'recall', 'f1_score']].add_prefix('pooled_'))
print('Equal-weight monthly mean ± std, and pooled over all four months '
      '(tuned models use their CV threshold, defaults use 0.5):')
display(summary.round(3))
display(pooled.round(3))

Actual late rate per month (a random model scores PR-AUC equal to this):


month,2018-04,2018-05,2018-06,2018-07
late_rate,0.066,0.082,0.022,0.052


PR-AUC by month:


month                        2018-04  2018-05  2018-06  2018-07
model               variant                                    
Decision Tree       default    0.091    0.101    0.041    0.082
                    tuned      0.091    0.104    0.039    0.074
LightGBM            default    0.114    0.147    0.089    0.125
                    tuned      0.123    0.146    0.098    0.120
Logistic Regression default    0.112    0.106    0.051    0.077
                    tuned      0.117    0.109    0.053    0.077
Random Forest       default    0.139    0.148    0.098    0.103
                    tuned      0.134    0.153    0.093    0.108
XGBoost             default    0.115    0.156    0.102    0.116
                    tuned      0.121    0.141    0.097    0.069

Equal-weight monthly mean ± std, and pooled over all four months (tuned models use their CV threshold, defaults use 0.5):


pr_auc       roc_auc       precision       recall  \
                              mean   std    mean   std      mean   std   mean   
model               variant                                                     
Decision Tree       default  0.079 0.026   0.570 0.016     0.070 0.026  0.476   
                    tuned    0.077 0.028   0.574 0.012     0.071 0.027  0.602   
LightGBM            default  0.119 0.024   0.604 0.033     0.081 0.018  0.495   
                    tuned    0.122 0.020   0.611 0.048     0.067 0.020  0.730   
Logistic Regression default  0.087 0.028   0.591 0.048     0.079 0.033  0.468   
                    tuned    0.089 0.030   0.594 0.047     0.075 0.042  0.550   
Random Forest       default  0.122 0.025   0.609 0.058     0.090 0.042  0.474   
                    tuned    0.122 0.026   0.609 0.055     0.077 0.036  0.582   
XGBoost             default  0.122 0.023   0.615 0.048     0.085 0.020  0.480   
                    tuned    0.107 0.031   0.602 0.060     0.059 0.026  0.828   

                                  f1_score        
                              std     mean   std  
model               variant                       
Decision Tree       default 0.266    0.120 0.048  
                    tuned   0.220    0.126 0.047  
LightGBM            default 0.214    0.139 0.034  
                    tuned   0.215    0.122 0.037  
Logistic Regression default 0.065    0.132 0.049  
                    tuned   0.196    0.124 0.057  
Random Forest       default 0.137    0.147 0.057  
                    tuned   0.116    0.135 0.057  
XGBoost             default 0.244    0.141 0.036  
                    tuned   0.163    0.110 0.047

pooled_pr_auc  pooled_precision  pooled_recall  \
model               variant                                                   
Decision Tree       default          0.087             0.079          0.573   
                    tuned            0.088             0.078          0.684   
LightGBM            default          0.129             0.088          0.575   
                    tuned            0.128             0.073          0.811   
Logistic Regression default          0.082             0.079          0.450   
                    tuned            0.082             0.070          0.560   
Random Forest       default          0.116             0.091          0.504   
                    tuned            0.118             0.081          0.611   
XGBoost             default          0.131             0.088          0.572   
                    tuned            0.109             0.064          0.888   

                             pooled_f1_score  
model               variant                   
Decision Tree       default            0.139  
                    tuned              0.140  
LightGBM            default            0.153  
                    tuned              0.134  
Logistic Regression default            0.135  
                    tuned              0.125  
Random Forest       default            0.154  
                    tuned              0.143  
XGBoost             default            0.152  
                    tuned              0.120

# Findings (this run; seeds fixed)
1. **A random split badly overstates performance.** For default LightGBM, stratified random 5-fold CV gives PR-AUC **0.43**, versus **0.29** for time-series CV. ROC-AUC is 0.76 versus 0.64. The random stratified splits in `model_classification_dev.ipynb` and `classification_evaluation.ipynb` are therefore optimistic. The report should quote the time-series numbers.
2. **Tuning helps only a little.**
   - CV PR-AUC improves by 0.00–0.03. That is smaller than the fold-to-fold standard deviation (≈ 0.07–0.09).
   - Best tuned CV PR-AUC: LightGBM 0.306 and XGBoost 0.305, then Random Forest 0.299. Logistic Regression and Decision Tree both reach 0.258.
   - The tree-based family beats the linear family by about 0.05 PR-AUC.
3. **In the April–July backtest, tuned ≈ default.**
   - LightGBM: 0.122 tuned vs 0.119 default. Random Forest: 0.122 for both.
   - Tuned XGBoost is worse: 0.107 vs 0.122.
   - The gains seen in CV largely don't carry into later months. That fits the small grids and the noisy, shifting target. It's a legitimate result to report.
4. **An F1-optimal probability threshold doesn't transfer across months.**
   - The out-of-fold thresholds (0.36–0.47) flag more orders and *lower* F1 on the hold-out and the backtest.
   - The cause is that the late rate drifts. It is ~26% in the last CV fold (late Feb to mid Mar 2018), ~10% in the hold-out and lower again in June.
   - Ranking quality (PR-AUC) carries over far better than any fixed cut-off does. Recommendation: set the late flag with a **capacity rule** (e.g. the top 10% riskiest orders each day, as in `classification_eval.daily_capacity`), not a probability threshold.
5. **Suggested final set, within the family budget:**
   - Logistic Regression (Linear family; plain LR is its own baseline).
   - Decision Tree → Random Forest / LightGBM (Tree-based family).
   - Use tuned LightGBM or Random Forest as the main model, and state that tuning gains are within noise.

# Save tuned parameters and thresholds

In [12]:
os.makedirs('results', exist_ok=True)
tuning_output = {
    'run_date': RUN_DATE, 'cv': f'{N_CV_FOLDS}-fold day-blocked expanding window',
    'scoring': 'average_precision',
    'models': {name: {'family': MODEL_FAMILY[name], 'best_params': params_to_json(best_params[name]),
                      'f1_threshold': thresholds[name],
                      'cv_pr_auc_default': cv_results.loc[(name, 'default'), 'cv_pr_auc_mean'],
                      'cv_pr_auc_tuned': cv_results.loc[(name, 'tuned'), 'cv_pr_auc_mean']}
               for name in tuned_models},
}
with open('results/classification_tuning.json', 'w') as f:
    json.dump(tuning_output, f, indent=2)
print(json.dumps(tuning_output, indent=2)[:1500])

{
  "run_date": "2018-06-02",
  "cv": "5-fold day-blocked expanding window",
  "scoring": "average_precision",
  "models": {
    "Logistic Regression": {
      "family": "Linear",
      "best_params": {
        "classifier__C": 100.0
      },
      "f1_threshold": 0.4533616786598222,
      "cv_pr_auc_default": 0.24422918464538562,
      "cv_pr_auc_tuned": 0.2576067853261545
    },
    "Decision Tree": {
      "family": "Tree-based",
      "best_params": {
        "classifier__min_samples_leaf": 50,
        "classifier__max_depth": 8,
        "classifier__criterion": "gini"
      },
      "f1_threshold": 0.4673005892801751,
      "cv_pr_auc_default": 0.22937753985414266,
      "cv_pr_auc_tuned": 0.2584938295672641
    },
    "Random Forest": {
      "family": "Tree-based",
      "best_params": {
        "classifier__n_estimators": 100,
        "classifier__min_samples_leaf": 5,
        "classifier__max_features": "sqrt",
        "classifier__max_depth": 10
      },
      "f1_threshold":

# Using the results in `model_classification_dev.ipynb`
```python
import json
tuning = json.load(open('results/classification_tuning.json'))
lgb_pipeline.set_params(**tuning['models']['LightGBM']['best_params'])
PREDICTION_THRESHOLD = tuning['models']['LightGBM']['f1_threshold']
```
The parameter names (`classifier__...`) match the dev notebook's pipeline step names.

# Notes for the report
- **Model family budget.** The brief allows 2–3 families. Here, Logistic Regression is the *Linear* family, and Decision Tree, Random Forest, XGBoost and LightGBM are all *Tree-based*. Each family starts from its simplest variant (plain LR; a single decision tree). XGBoost and LightGBM are close alternatives, so the report could keep one and put the other in the appendix.
- **Tuning method.** Random search over small grids on 5 expanding-window folds, scored by PR-AUC. The threshold comes from out-of-fold F1. Seeds are fixed (`RANDOM_STATE = 42`), and all preprocessing is fitted inside each pipeline and fold.
- **Evaluation.** Report CV mean ± std, hold-out metrics, and the April–July backtest (monthly mean ± std and pooled). The backtest months come after all data used for tuning.
- **Limitations.**
  - Grids are small; widen them once the features are frozen.
  - The threshold is tuned for F1, not for a business cost or capacity.
  - Scores are not calibrated probabilities.